# Imports

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Base Variables

In [ ]:
is_robust = True

if is_robust:
    TRAIN_JSONL_PATH = "../../data/robust_vqa_split/vqa_train.jsonl"
    VAL_JSONL_PATH = "../../data/robust_vqa_split/vqa_val.jsonl"
    TEST_JSONL_PATH = "../../data/robust_vqa_split/vqa_test.jsonl"
else:
    TRAIN_JSONL_PATH = "../../data/base_vqa_split/vqa_train.jsonl"
    VAL_JSONL_PATH = "../../data/base_vqa_split/vqa_val.jsonl"
    TEST_JSONL_PATH = "../../data/base_vqa_split/vqa_test.jsonl"

DEFAULT_IMAGE_FIELD = "image_path"

TOPIC_COL = "task_type"

# Auxiliary Functions

In [ ]:
def load_jsonl(path: str | Path) -> pd.DataFrame:
    return pd.read_json(path, lines=True)


def load_image(path: str | Path) -> np.ndarray:
    path = "../" + str(path)

    if path.endswith(".npy"):
        img = np.load(path)
    else:
        raise ValueError(f"Unsupported image format for this helper: {path}")

    # squeeze singleton channel dimensions
    if img.ndim == 3 and img.shape[0] == 1:
        img = img[0]
    elif img.ndim == 3 and img.shape[-1] == 1:
        img = img[..., 0]

    return img

def format_multi_turn_prompt(example: dict) -> str:
    lines = [f"Context: {example['context']}"]
    for turn in example["conversation"]:
        role = turn["role"].capitalize()
        lines.append(f"{role}: {turn['text']}")
    return "\n".join(lines)


def format_multi_turn_pretty(example: dict) -> str:
    lines = []
    lines.append("=== CONTEXT ===")
    lines.append(example["context"])
    lines.append("")
    lines.append("=== CONVERSATION ===")
    for i, turn in enumerate(example["conversation"], start=1):
        role = turn["role"].capitalize()
        lines.append(f"{i:02d}. {role}: {turn['text']}")
    return "\n".join(lines)

def preview_turn(df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    cols = [
        c for c in [
            "id",
            "source_id",
            "split",
            "field_id",
            "task_type",
            "question",
            "answer",
            "image_path",
        ]
        if c in df.columns
    ]
    return df[cols].head(n)

def show_multi_turn_example(
    df: pd.DataFrame,
    row_index: int = 0,
    image_field: str = DEFAULT_IMAGE_FIELD,
    figsize: tuple[int, int] = (6, 6),
) -> dict:
    example = df.iloc[row_index].to_dict()

    print("Example ID:", example.get("id"))
    print("Source ID:", example.get("source_id"))
    print("Split:", example.get("split"))
    print()

    image_path = example.get(image_field)
    if image_path is None:
        raise KeyError(f"Field '{image_field}' not found in example.")

    img = load_image(image_path)

    plt.figure(figsize=figsize)
    plt.imshow(img, cmap="gray")
    plt.axis("off")
    plt.show()

    print(format_multi_turn_pretty(example))
    print()
    print("=== RAW MODEL INPUT STYLE ===")
    print(format_multi_turn_prompt(example))

    return example

def load_dataset(path: str | Path) -> pd.DataFrame:
    return load_jsonl(path)


def inspect_multi_turn(
    path: str | Path,
    row_index: int = 0,
    image_field: str = DEFAULT_IMAGE_FIELD,
) -> tuple[pd.DataFrame, dict]:
    df = load_dataset(path)
    display(df, n=5)
    example = show_multi_turn_example(df, row_index=row_index, image_field=image_field)
    return df, example

def show_multi_turn_examples_range(
    df: pd.DataFrame,
    start: int = 0,
    end: int = 3,
    image_field: str = DEFAULT_IMAGE_FIELD,
):
    for idx in range(start, min(end, len(df))):
        print(f"\n{'='*100}\nMULTI-TURN ROW {idx}\n{'='*100}")
        show_multi_turn_example(df, row_index=idx, image_field=image_field)

In [ ]:
def plot_question_counts(df: pd.DataFrame, col: str = "task_type", figsize=(12, 6)):
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="count")
    )

    display(counts)

    plt.figure(figsize=figsize)
    plt.bar(counts[col], counts["count"])
    plt.xticks(rotation=45, ha="right")
    plt.xlabel(col)
    plt.ylabel("Number of questions")
    plt.title(f"Number of questions per {col}")
    plt.tight_layout()
    plt.show()

    return counts

# Turn Analysis

In [ ]:
df = load_dataset(TEST_JSONL_PATH)
df

In [ ]:
df["conversation"].iloc[0]

In [ ]:
preview_turn(df, 5)

In [ ]:
df.iloc[0]

In [ ]:
og_df = pd.read_csv("../../data/report_generation_split/all-rg-test.csv")
og_df.shape

In [ ]:
og_df[og_df["id"] == "5145e778-3cda-4cc9-a655-6b74ee4eea67"]["findings"].values

In [ ]:
show_multi_turn_example(df, row_index=0)

# Granular Analysis

In [ ]:
df = pd.read_json(VAL_JSONL_PATH, lines=True)

In [ ]:
main_df = pd.read_csv("../../data/report_generation_split/all-rg-val.csv")
main_df["_memmap_idx"] = np.arange(len(main_df), dtype=np.int64)

In [ ]:
main_df.head(2)

In [ ]:
out = df.merge(main_df, left_on="source_id", right_on="id", how="left", suffixes=("_vqa", "_rg"))

In [ ]:
merge_cols = [
    "id",
    "_memmap_idx",
    "birads",
    "original_birads",
    "modality",
    "dataset",
    "patient",
]
out = df.merge(
    main_df[merge_cols],
    left_on="source_id",
    right_on="id",
    how="left",
    suffixes=("_vqa", "_rg"),
)

In [ ]:
out.head(2)

In [ ]:
out["original_birads"].value_counts(dropna=False)

In [ ]:
import json

bad_lines = []
with open(TRAIN_JSONL_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        try:
            json.loads(line)
        except Exception as e:
            bad_lines.append((i, str(e)))

print("Bad lines:", bad_lines[:10])
print("Total bad lines:", len(bad_lines))

# Count number of negative examples across sets

In [ ]:
train_df = load_dataset(TRAIN_JSONL_PATH)
val_df = load_dataset(VAL_JSONL_PATH)
test_df = load_dataset(TEST_JSONL_PATH)

In [ ]:
train_total, train_negatives = 0,0
test_total, test_negatives = 0,0
val_total, val_negatives = 0,0

In [ ]:
for _, elem in train_df.iterrows():
    conversation = elem["conversation"]
    for elem in conversation:
        train_total += 1
        if elem["role"] == "assistant":
            train_negatives += int(elem.get("is_negative", False))

In [ ]:
train_negatives, train_total

In [ ]:
701744/3257866*100

In [ ]:
for _, elem in test_df.iterrows():
    conversation = elem["conversation"]
    for elem in conversation:
        test_total += 1
        if elem["role"] == "assistant":
            test_negatives += int(elem.get("is_negative", False))

test_negatives, test_total

In [ ]:
for _, elem in val_df.iterrows():
    conversation = elem["conversation"]
    for elem in conversation:
        val_total += 1
        if elem["role"] == "assistant":
            val_negatives += int(elem.get("is_negative", False))

val_negatives, val_total

# Count the number of samples total, and also per scope

In [ ]:
count_questions = {}

try:

    for _, elem in train_df.iterrows():
        conversation = elem["conversation"]
        for elem in conversation:
            if elem["role"] == "user":
                continue
            if elem["role"] == "assistant":
                question_type = elem.get("task_type", "unknown")
                if question_type not in count_questions:
                    count_questions[question_type] = 0 if question_type != "assessment_birads" else {}
                
                if question_type == "assessment_birads":
                    answer = elem.get("text", "")
                    if answer not in count_questions[question_type]:
                        count_questions[question_type][answer] = 0
                    count_questions[question_type][answer] += 1
                else:
                    count_questions[question_type] += 1
except Exception as e:
    print("Error processing conversations:", str(e), question_type, answer)

In [ ]:
count_questions

In [ ]:
train_df